# Sprint 1: Next-Game Points Prediction Proof of Concept

This notebook inspects the synthetic sample dataset and illustrates the leakage-safe target definition. It intentionally does **not** train a machine-learning model, call an NBA API, or calculate betting probabilities.


## Prediction design

A row at game *t* uses completed history through that game to predict points in game *t + 1*. The target is therefore the same player's next chronological points value. Statistical feature calculations must never include the target game or later games.

The future feature set is: season points average, last-5 points average, previous-game points, recent minutes average, and known home/away context.


In [1]:
from pathlib import Path
import pandas as pd

# Locate the CSV in a local checkout or common hosted-notebook mounts.
DATA_NAME = 'sample_nba_games.csv'
DIRECT_ROOTS = (Path.cwd(), *Path.cwd().parents)
HOSTED_ROOTS = (Path('/mnt/data'), Path('/home/jovyan'), Path('/home/oai/share'), Path('/workspace'), Path('/workspaces'), Path('/content'))
DATA_PATH = next((root / 'data' / DATA_NAME for root in DIRECT_ROOTS if (root / 'data' / DATA_NAME).is_file()), None)

# Hosted notebooks may mount the repository one level below one of these folders.
if DATA_PATH is None:
    for root in HOSTED_ROOTS:
        if root.is_dir():
            matches = list(root.glob(f'**/data/{DATA_NAME}'))
            if matches:
                DATA_PATH = matches[0]
                break

if DATA_PATH is None:
    raise FileNotFoundError(
        f'Could not find data/{DATA_NAME} from {Path.cwd()}. Ensure the repository data folder is available to this Jupyter kernel.'
    )

games = pd.read_csv(DATA_PATH, parse_dates=['game_date'])
games = games.sort_values(['player_id', 'game_date']).reset_index(drop=True)

print(f'Loaded {len(games)} synthetic game rows for {games.player_id.nunique()} players from {DATA_PATH}')
games.head()


Loaded 152 synthetic game rows for 8 players from /Users/swayamshree/Desktop/Capstone Project/CPSC491-02Group8SportsBetting/data/sample_nba_games.csv


,player_id,player_name,game_date,team,opponent,home_away,minutes,points,rebounds,assists,field_goal_attempts
0,1,LeBron James,2025-01-03,LAL,MIA,HOME,35,29,6,8,17
1,1,LeBron James,2025-01-06,LAL,CHI,AWAY,33,22,8,9,19
2,1,LeBron James,2025-01-09,LAL,CLE,HOME,35,27,7,7,18
3,1,LeBron James,2025-01-12,LAL,ATL,AWAY,37,32,9,10,21
4,1,LeBron James,2025-01-15,LAL,ORL,HOME,34,26,5,6,16


In [2]:
# Validation only: each player's dates must be chronological.
is_chronological = games.groupby('player_id')['game_date'].apply(lambda dates: dates.is_monotonic_increasing)
assert is_chronological.all()
assert games['minutes'].between(20, 40).all()
assert games['points'].between(5, 45).all()
assert games['rebounds'].between(0, 15).all()
assert games['assists'].between(0, 12).all()
games.groupby(['player_id', 'player_name']).size().rename('games').reset_index()


,player_id,player_name,games
0,1,LeBron James,19
1,2,Stephen Curry,19
2,3,Nikola Jokic,19
3,4,Luka Doncic,19
4,5,Jayson Tatum,19
5,6,Jalen Brunson,19
6,7,Mikal Bridges,19
7,8,Herb Jones,19


In [3]:
# Prototype target definition only; this is not a training dataset or a trained model.
# For game t, next_game_points is the value to predict for game t + 1.
prototype = games[['player_id', 'player_name', 'game_date', 'points']].copy()
prototype['next_game_points'] = prototype.groupby('player_id')['points'].shift(-1)
prototype['previous_game_points'] = prototype.groupby('player_id')['points'].shift(1)

# The final game for each player has no following-game target in this sample.
prototype.dropna(subset=['next_game_points']).head(10)


,player_id,player_name,game_date,points,next_game_points,previous_game_points
0,1,LeBron James,2025-01-03,29,22.0,NaN
1,1,LeBron James,2025-01-06,22,27.0,29.0
2,1,LeBron James,2025-01-09,27,32.0,22.0
3,1,LeBron James,2025-01-12,32,26.0,27.0
4,1,LeBron James,2025-01-15,26,30.0,32.0
5,1,LeBron James,2025-01-18,30,21.0,26.0
6,1,LeBron James,2025-01-21,21,28.0,30.0
7,1,LeBron James,2025-01-24,28,26.0,21.0
8,1,LeBron James,2025-01-27,26,33.0,28.0
9,1,LeBron James,2025-01-30,33,23.0,26.0


## Planned evaluation

Use older games for training and newer games for testing (approximately 80/20). Start with the last-5-games points average as a baseline; later compare Linear Regression or Random Forest only after the baseline is measured on the same chronological split.
